# Shepherd health-monitor analysis

List one or more session `.h5` files and point at your shepherd-log folder, then **Run All**:

- **`H5_PATHS`** — the session `.h5` file(s); each defines a session window + camera-fps facts
- **`LOGS_DIR`** — the folder holding the shepherd logs (`shepherd_*.jsonl` + `shepherd_*.alerts.log`)

**The process-health data is not in the `.h5`.** shepherd runs as a separate watchdog on the leader
and writes it to `~/shepherd_logs/`; the `.h5` files only tell this notebook *when* each session ran.
Because shepherd runs continuously, one log usually covers every session from the same day. Copy the
logs across with e.g.
`scp vruser@192.168.10.101:'~/shepherd_logs/shepherd_*' ~/Downloads/shepherd_logs/`.

The notebook picks the newest log in `LOGS_DIR`, puts every metric on one timeline spanning the listed
sessions, shades each session in its own colour, draws the shepherd warn/critical thresholds, and ticks
the alerts that fired — then a per-session summary + the in-window alerts.

In [ ]:
from pathlib import Path
import json, re
from datetime import datetime, timezone
import numpy as np, pandas as pd, h5py
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# ── Point these at your session(s) — list one or more .h5 files ────────────
H5_PATHS = [
    Path("/Users/hakan/Downloads/ASD109/ASD109_20260819/ASD109_20260819_001.h5"),   # earlier session
    Path("/Users/hakan/Downloads/ASD110/ASD110_20260819/ASD110_20260819_001.h5"),   # later session
]
LOGS_DIR = Path("/Users/hakan/Downloads/shepherd_logs")    # folder with shepherd_*.jsonl + .alerts.log

# shepherd/config.yaml warn/critical thresholds — the dashed guide lines on each panel
THRESHOLDS = {
    "soc_temp_c":   dict(warn=50, crit=70),
    "cpu_percent":  dict(warn=85, crit=95),
    "mem_percent":  dict(warn=85, crit=95),
    "disk_free_gb": dict(warn=40, crit=15),
    "camera_fps":   dict(warn=45, crit=30),
}
PAD_MIN = 10                                                # minutes of context around the session span
SESSION_COLORS = ["#3d7fe8", "#8e44ad", "#159e73", "#d67f2c"]   # one tint per session

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.axisbelow": True, "font.size": 10})
WARN_C, CRIT_C = "#e8a33d", "#d64545"

In [ ]:
# ── pick the shepherd log (newest in LOGS_DIR) ─────────────────────────────
jsonls = sorted(LOGS_DIR.glob("shepherd_*.jsonl"))
assert jsonls, f"no shepherd_*.jsonl in {LOGS_DIR} — copy them from the leader (~/shepherd_logs/)"
JSONL_PATH  = jsonls[-1]
ALERTS_PATH = JSONL_PATH.parent / (JSONL_PATH.stem + ".alerts.log")   # shepherd_X.jsonl -> shepherd_X.alerts.log

# ── shepherd metrics (one ~1 Hz sample per line) ───────────────────────────
records = [json.loads(l) for l in JSONL_PATH.read_text().splitlines() if l.strip()]
df = pd.json_normalize(records, sep="_").sort_values("t").reset_index(drop=True)  # nested api/throttled -> api_*, throttled_*
t0, tN = float(df["t"].iloc[0]), float(df["t"].iloc[-1])
print(f"log     : {JSONL_PATH.name}  ({len(df)} samples, {(tN-t0)/60:.0f} min — shepherd runs continuously)")

# ── session windows + camera facts (one per .h5) ───────────────────────────
def session_window(h5_path):
    out = {"path": h5_path, "window": None, "measured_fps": np.nan, "subject": "?", "sess_id": "?"}
    with h5py.File(h5_path, "r") as f:
        if "trials/running_t" in f:
            rt = f["trials/running_t"][()]
            out["window"] = (float(rt.min()), float(rt.max()))
        elif "camera/frame_timestamps" in f:
            wc = f["camera/frame_timestamps"][()][:, 1]; wc = wc[wc > 0]
            if wc.size:
                out["window"] = (float(wc.min()), float(wc.max()))
        if "camera" in f:
            out["measured_fps"] = float(f["camera"].attrs.get("measured_fps", np.nan))
        out["subject"] = f.attrs.get("subject_id", f.attrs.get("subject", "?"))
        out["sess_id"] = f.attrs.get("session_id", "?")
    return out

sessions = [session_window(p) for p in H5_PATHS]
sessions = [s for s in sessions if s["window"]]            # keep only those with a usable window
sessions.sort(key=lambda s: s["window"][0])               # chronological
assert sessions, "no .h5 has a session window (trials/running_t or camera timestamps)"

x0 = sessions[0]["window"][0]                             # x-axis origin: first session start
df["min"] = (df["t"] - x0) / 60.0                         # minutes since the first session started
for i, s in enumerate(sessions):
    s0, s1 = s["window"]
    s["sm0"], s["sm1"] = (s0 - x0) / 60.0, (s1 - x0) / 60.0
    s["color"] = SESSION_COLORS[i % len(SESSION_COLORS)]
    flag = "" if (t0 <= s0 <= tN) else "   ** NOT in this log — copy the shepherd_*.jsonl that covers it"
    print(f"session : {s['subject']} {s['sess_id']}  {(s1-s0)/60:.1f} min, "
          f"{(s0-t0)/60:.0f} min into the log{flag}")

# ── alerts (leader-local ISO -> unix, anchored on the JSONL launch stamp) ───
# The .jsonl filename carries the launch time in the LEADER's local clock; its first sample carries
# the matching unix time. Parsing the stamp as UTC and subtracting gives the leader's tz offset, so
# alert times land correctly even when this notebook runs in another timezone.
def _leader_offset():
    m = re.search(r"(\d{8}_\d{6})", JSONL_PATH.name)
    if not m:
        return None
    launch = datetime.strptime(m.group(1), "%Y%m%d_%H%M%S").replace(tzinfo=timezone.utc).timestamp()
    return launch - t0

_off = _leader_offset()
def _iso_to_unix(ts_s):
    naive = datetime.fromisoformat(ts_s)
    return naive.timestamp() if _off is None else naive.replace(tzinfo=timezone.utc).timestamp() - _off

def load_alerts(path):
    rows = []
    for line in Path(path).read_text().splitlines():
        line = line.strip()
        if not line:
            continue
        try:                                               # "2026-08-19T09:08:46 [LEVEL] metric: message"
            ts_s, rest = line.split(" ", 1)
            ts = _iso_to_unix(ts_s)
            level = rest[rest.index("[") + 1:rest.index("]")].lower().strip()
            after = rest[rest.index("]") + 1:].strip()
            metric, msg = (after.split(":", 1) + [""])[:2]
            rows.append(dict(t=ts, min=(ts - x0) / 60.0, level=level,   # x0: first session start
                             metric=metric.strip(), message=msg.strip()))
        except Exception:
            continue
    return pd.DataFrame(rows)

alerts = load_alerts(ALERTS_PATH) if ALERTS_PATH.exists() else pd.DataFrame()
print(f"alerts  : {len(alerts)} lines  ({ALERTS_PATH.name})")

In [ ]:
# One panel per metric, shared timeline. (col, title, y-label, threshold/alert key)
PANELS = [
    ("soc_temp_c",     "SoC temperature",   "°C", "soc_temp_c"),
    ("cpu_percent",    "CPU load",          "%",       "cpu_percent"),
    ("load1",          "Load (1 min)",      "load",    None),
    ("mem_percent",    "Memory used",       "%",       "mem_percent"),
    ("disk_free_gb",   "Disk free",         "GB",      "disk_free_gb"),
    ("api_camera_fps", "Camera encode fps", "fps",     "camera_fps"),
]

# Span all listed sessions (+ PAD_MIN of context); the log itself covers days of idle time.
lo = min(s["sm0"] for s in sessions) - PAD_MIN
hi = max(s["sm1"] for s in sessions) + PAD_MIN
view = df[(df["min"] >= lo) & (df["min"] <= hi)]

fig, axes = plt.subplots(len(PANELS), 1, figsize=(11, 2.0 * len(PANELS)), sharex=True)
for ax, (col, title, ylab, key) in zip(axes, PANELS):
    if col in view:
        d = view[["min", col]].dropna()                    # api_camera_fps is null off-recording -> drops out
        ax.plot(d["min"], d[col], color="#2b6cb0", lw=1.2)
    ax.set_ylabel(ylab)
    ax.set_title(title, loc="left", fontsize=10, fontweight="bold")
    if ylab == "%":
        ax.set_ylim(0, 100)
    for s in sessions:                                     # shade each session in its own colour
        ax.axvspan(s["sm0"], s["sm1"], color=s["color"], alpha=0.10, zorder=0)
    sp = THRESHOLDS.get(key, {})
    if sp.get("warn") is not None:
        ax.axhline(sp["warn"], color=WARN_C, ls="--", lw=1, alpha=0.8)
    if sp.get("crit") is not None:
        ax.axhline(sp["crit"], color=CRIT_C, ls="--", lw=1, alpha=0.8)
    if col == "api_camera_fps":                            # measured fps reference (per distinct value)
        for v in sorted({round(s["measured_fps"], 1) for s in sessions if np.isfinite(s["measured_fps"])}):
            ax.axhline(v, color="#2b6cb0", ls=":", lw=1, alpha=0.6)
    if key and len(alerts):                                # tick this panel's alerts (in view) at the top
        vis = alerts[(alerts["metric"] == key) & (alerts["min"] >= lo) & (alerts["min"] <= hi)]
        ymax = ax.get_ylim()[1]
        for _, a in vis.iterrows():
            ax.plot(a["min"], ymax, marker="v", ms=6, clip_on=False, zorder=5,
                    color=CRIT_C if a["level"] == "critical" else WARN_C)

axes[-1].set_xlim(lo, hi)
axes[-1].set_xlabel("minutes since first session start")
fig.legend(handles=[Patch(facecolor=s["color"], alpha=0.30, label=f"{s['subject']} {s['sess_id']}")
                    for s in sessions],
           loc="upper right", fontsize=8, framealpha=0.9)
fig.suptitle("Shepherd health — " + " · ".join(s["subject"] for s in sessions) + "  (sessions shaded)",
             fontsize=12, fontweight="bold", y=0.997)
fig.tight_layout()
plt.show()

In [ ]:
# ── per-session summary + the alerts that fired in each window ──────────────
for s in sessions:
    s0, s1 = s["window"]
    sdf = df[(df["t"] >= s0) & (df["t"] <= s1)]
    cols = [c for c in ["soc_temp_c", "cpu_percent", "mem_percent", "disk_free_gb", "api_latency_ms"] if c in sdf]
    print(f"=== {s['subject']} {s['sess_id']}  ({(s1-s0)/60:.1f} min) ===")
    display(sdf[cols].astype(float).agg(["min", "median", "max"]).T.round(2))
    if len(alerts):
        insess = alerts[(alerts["t"] >= s0) & (alerts["t"] <= s1)]
        print(f"{len(insess)} alert(s) during this session:")
        if len(insess):
            display(insess[["min", "level", "metric", "message"]].reset_index(drop=True))
    print()